# Concept scores: review

Scores 0–10 that Gemini gave products for every concept, read from the scoring logs in
`search_ui/data/concept_scores/` (`scores.jsonl`: the current prompt with concept descriptions and reference points from
several categories; `pilot_minimal.jsonl`: the older prompt, for comparison on the same dresses).

- **Summary** – per category and concept: mean, share of 7 or more (should usually stay under a fifth), share of 9–10.
- **Old vs new prompt** – the same dresses scored by both prompts.
- **Browse** – `top("Jackets", "rainy_weather")` and `bottom(...)` show products with their photos and scores;
  `product("<id>")` shows every concept score of one product.

In [ ]:
import sys, json, base64, html
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import HTML, display

HERE = Path.cwd() if (Path.cwd() / "evaluate.py").exists() else Path.cwd() / "search_ui/evaluation"
sys.path[:0] = [str(HERE.parent / "core")]
import common, concepts

LOGS = common.DATA / "concept_scores"
def load(name):
    rows = []
    for r in map(json.loads, (LOGS / name).open(encoding="utf-8")):
        for cid, score in (r.get("scores") or {}).items():
            rows.append(dict(product_id=r["product_id"], concept_id=cid, score=score))
    return pd.DataFrame(rows).drop_duplicates(["product_id", "concept_id"], keep="last")

new = load("scores.jsonl")
old = load("pilot_minimal.jsonl")
products, _ = common.load_index()
by_id = {p["product_id"]: p for p in products}
names = {c["id"]: c["name"] for c in concepts.load_list()}
new["category"] = new.product_id.map(lambda i: by_id[i]["category"])
print(f"new prompt: {new.product_id.nunique()} products, {new.concept_id.nunique()} concepts; "
      + ", ".join(f"{c} {n}" for c, n in new.groupby('category').product_id.nunique().items()))

## Summary per category and concept

In [ ]:
def summary(category):
    d = new[new.category == category]
    t = d.groupby("concept_id").score.agg(mean="mean", share_7=lambda s: (s >= 7).mean(), share_9=lambda s: (s >= 9).mean())
    t.index = [names[i] for i in t.index]
    return t.sort_values("mean", ascending=False).style.format({"mean": "{:.1f}", "share_7": "{:.0%}", "share_9": "{:.0%}"})

display(summary("Jackets"))

In [ ]:
display(summary("Knit Sweaters"))

## Old vs new prompt on the same dresses

In [ ]:
both = new.merge(old, on=["product_id", "concept_id"], suffixes=("_new", "_old"))
print(f"{both.product_id.nunique()} dresses scored by both prompts")
t = both.groupby("concept_id").apply(lambda g: pd.Series(dict(
    old_share_7=(g.score_old >= 7).mean(), new_share_7=(g.score_new >= 7).mean(),
    old_share_9=(g.score_old >= 9).mean(), new_share_9=(g.score_new >= 9).mean(),
    spearman=g[["score_old", "score_new"]].corr(method="spearman").iloc[0, 1])))
t.index = [names[i] for i in t.index]
print(f"all pairs: share of 7+ old {(both.score_old >= 7).mean():.0%} -> new {(both.score_new >= 7).mean():.0%}; "
      f"9-10 old {(both.score_old >= 9).mean():.0%} -> new {(both.score_new >= 9).mean():.0%}; "
      f"Spearman {both[['score_old', 'score_new']].corr(method='spearman').iloc[0, 1]:.2f}")
t.sort_values("old_share_7", ascending=False).style.format("{:.2f}", subset=["spearman"]).format("{:.0%}", subset=t.columns[:4].tolist())

## Browse products

In [ ]:
_images = {}
def thumb(pid):
    if pid not in _images:
        p = by_id[pid]
        path = next((x for x in p["images"] if "id_gridfs_1." in x), p["images"][0] if p["images"] else None)
        _images[pid] = base64.b64encode((common.DATASET / path).read_bytes()).decode() if path else ""
    return _images[pid]

def grid(rows, title):
    cells = "".join(
        f"<div style='width:130px;font-size:11px'><img src='data:image/png;base64,{thumb(r.product_id)}' "
        f"style='width:130px;height:130px;object-fit:contain;background:#fff'><br><b>{r.score}</b> "
        f"{html.escape(by_id[r.product_id]['name'][:40])}</div>" for r in rows.itertuples())
    display(HTML(f"<h4>{html.escape(title)}</h4><div style='display:flex;flex-wrap:wrap;gap:6px'>{cells}</div>"))

def top(category, concept, n=10):
    d = new[(new.category == category) & (new.concept_id == concept)].sort_values("score", ascending=False)
    grid(d.head(n), f"{category} · {names[concept]} · highest")

def bottom(category, concept, n=10):
    d = new[(new.category == category) & (new.concept_id == concept)].sort_values("score")
    grid(d.head(n), f"{category} · {names[concept]} · lowest")

def product(pid):
    d = new[new.product_id == pid].sort_values("score", ascending=False)
    grid(d.head(1), by_id[pid]["name"])
    print(", ".join(f"{names[r.concept_id]} {r.score}" for r in d.itertuples()))

In [ ]:
top("Jackets", "rainy_weather")

In [ ]:
top("Jackets", "winter")

In [ ]:
top("Knit Sweaters", "cozy")

In [ ]:
top("Knit Sweaters", "christmas")

In [ ]:
top("Jackets", "office"); bottom("Jackets", "office")